# Transfer functions

Atmospheric emission can dominate the noise budget and must be removed from the time-ordered data before mapping. This filtering inevitably projects out correlated sky modes, suppressing signal at large angular scales. The spatial transfer function $T(u)$ provides a quantification of this scale-dependent signal loss. `maria` provides a built-in function to estimate this, using the cross-spectrum between the output and the known input map:

$$T(u) = \frac{\mathrm{Re}\langle \tilde{s}_{\rm in}^*(u)\,\tilde{s}_{\rm out}(u)\rangle}{\langle|\tilde{s}_{\rm in}(u)|^2\rangle}$$

The cross-spectrum is used instead of the power ratio $P_{\rm out}/P_{\rm in}$ because noise in the output is uncorrelated with the input and averages to zero, leaving an unbiased estimate.

## Input sky

In this tutorial, we simulate a two-frequency observation of a galaxy cluster at 150 and 300 GHz, using a two-band instrument on a 50-metre primary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import maria
from maria.io import fetch

map_filename = fetch("maps/cluster2.fits")

nu = np.array([150e9, 300e9])

m1 = maria.map.load(filename=map_filename, nu=nu[0])
m2 = maria.map.load(filename=map_filename, nu=nu[1])

for m in [m1, m2]:
    m.data *= 1e2

maps = maria.map.concatenate([m1, m2], dim="nu")
print(maps)

In [ ]:
maps.to("K_RJ").plot(slices=dict(nu=[0, 1]), cmap="RdYlBu_r")

In [ ]:
from maria.instrument import Band

f150 = Band(center=150e9, width=30e9, NET_RJ=40e-6, knee=1e0)
f300 = Band(center=300e9, width=50e9, NET_RJ=80e-6, knee=1e0)

array = {
    "field_of_view": 0.5,
    "beam_spacing": 1.5,
    "primary_size": 50,
    "bands": [f150, f300],
}

instrument = maria.get_instrument(array=array)

print(instrument)
instrument.plot()

In [ ]:
from maria import Planner

planner = Planner(
    target=maps,
    start_time="2024-01-01T22:00:00",
    site="llano_de_chajnantor",
    constraints={"el": (60, 90)},
)

plans = planner.generate_plans(
    total_duration=0.1 * 3600,
    max_chunk_duration=3600,
    scan_options={"radius": 6.5 / 60.0, "miss_factor": 0.3},
)

print(plans)
plans[0].plot()

## Simulation

Passing `map=maps` attaches the ground-truth sky to each output TOD so the mapper can propagate it to the output map automatically.

In [ ]:
sim = maria.Simulation(
    instrument,
    plans=plans,
    site="llano_de_chajnantor",
    map=maps,
    atmosphere="2d",
)

print(sim)

tods = sim.run()
tods[0].plot()

## Mapmaking

Common-mode subtraction (`remove_modes`) and per-detector spline removal (`remove_spline`) suppress large-scale correlated signal. Both operations remove power at long time scales, which maps to large angular scales on the sky — exactly where $T$ will fall below 1.

In [ ]:
from maria.mappers import BinMapper

mapper = BinMapper(
    tods=tods,
    units="uK_RJ",
    stokes="I",
    resolution=maps.resolution,
    tod_preprocessing={
        "remove_modes": {"modes_to_remove": 1},
        "remove_spline": {"knot_spacing": 60, "remove_el_gradient": True},
    },
    map_postprocessing={
        "gaussian_filter": {"sigma": 1},
    },
)

output_map = mapper.run()

In [ ]:
output_map.plot(slices=dict(nu=[0, 1]), cmap="RdYlBu_r")

## Transfer function

To compute the transfer function, call `transfer_function()` on the output map. Because `map=maps` was passed to the simulation, the input sky has been propagated automatically through the TODs to the output map — no extra arguments are needed.

In [ ]:
tf = output_map.transfer_function()
print(tf)

... Solid curves show the measured $T(u)$; dashed curves show the Gaussian beam per channel. The large-scale rolloff reflects the TOD filtering above; the small-scale rolloff tracks the beam. The 300 GHz cutoff sits at twice the spatial frequency of the 150 GHz one.

In [1]:
tf.plot(x_unit="arcmin")

NameError: name 'tf' is not defined

Solid curves show the measured $T(u)$; dashed curves show the Gaussian beam per channel. The large-scale rolloff reflects the TOD filtering; the small-scale rolloff tracks the beam. The 300 GHz cutoff sits at twice the spatial frequency of the 150 GHz one, consistent with the factor-of-two difference in beam size.

Individual channels can be selected with `slices=dict(nu=[...])`, consistent with how `slices` is used in `.plot()`:

In [ ]:
tf.plot(slices=dict(nu=[0, 2]), x_unit="arcmin")